In [67]:
ALPHAVANTAGE_API_KEY = 'Q4L3OK0NNB37YQQZ'

import pandas as pd
import numpy as np
import json
import glob

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

---
#### ExtraETF
---

In [68]:
# Load justETF json
with open('data/ibkr_justetfs_scraped.json', 'r') as f:
    url_data = json.load(f)

    data_list = []
    for url in url_data:
        data = {}
        for prop in url:
            if type(prop) == str:
                data['url'] = prop
            else:
                data.update(prop)
        data_list.append(data)
    df = pd.DataFrame(data_list)

df['isin'] = df['url'].apply(lambda x: x.split('=')[-1])
df = df[['url', 'isin', 'exchanges', 'índice','proveedor_de_fondo','fecha_de_inicio/_de_cotización', 'política_de_distribución', 'domicilio_del_fondo', 'ter', 'replicación', 'riesgo_estratégico', 'riesgo_de_divisa','sostenibilidad', 'divisa_del_fondo', 'tamaño_del_fondo', 'participaciones', 'holdings', 'top_10_holdings', 'historic_dividend', 'rentabilidad_actual_de_los_dividendos',  'frecuencia_de_distribución', 'dividendos_(últimos_12_meses)', 'volatilidad_1_año_(in_eur)', 'categoría', 'foco_de_la_inversión', 'sectors', 'countries', ]]

In [69]:
# explode sectors, countries, and historic_dividend
def expand_sectors(row):
    if row == []:
        return {}
    elif row == None:
        return {}
    else:
        return dict((f'{col}_{x[0]}', x[1]) for x in row)

list_columns = ['sectors', 'countries', 'historic_dividend',]

for col in list_columns:
    df[col] = df[col].apply(expand_sectors)
    temp_df = df[col].apply(pd.Series)
    df = df.drop([col], axis=1)

    df = pd.concat([df, temp_df], axis=1)

In [70]:
# Clean up df
for col in df.columns:
    try:
        df[col] = df[col].replace('-', np.nan)
        df[col] = df[col].apply(lambda x: np.nan if len(x)==0 else x)
    except TypeError:
        continue
    
df['rentabilidad_actual_de_los_dividendos'] = df['rentabilidad_actual_de_los_dividendos'].replace('dividendos_(últimos_12_meses)', np.nan)

df['sostenibilidad'] = df['sostenibilidad'].replace('Sí', True)
df['sostenibilidad'] = df['sostenibilidad'].replace('No', False)

df = df.dropna(subset=['exchanges'])

In [71]:
# correct up numeric values
non_numeric = ['url',	'exchanges', 'holdings', 'índice',	'foco_de_la_inversión', 'replicación',	'riesgo_estratégico', 'riesgo_de_divisa', 'sostenibilidad',	'divisa_del_fondo', 'fecha_de_inicio/_de_cotización',	'política_de_distribución',	'frecuencia_de_distribución',	'domicilio_del_fondo',	'proveedor_de_fondo',	'categoría', 'isin',]

numeric = [item for item in df.columns.to_list() if item not in non_numeric]

for col in numeric:
    df[col] = df[col].apply(lambda x: x.replace('.', '') if isinstance(x, str) else x)
    df[col] = df[col].apply(lambda x: x.replace(',', '.') if isinstance(x, str) else x)
    df[col] = df[col].apply(lambda x: x.replace('+', '') if isinstance(x, str) else x)
    df[col] = df[col].apply(lambda x: x.replace(' m', '000000') if isinstance(x, str) else x)
    df[col] = df[col].apply(lambda x: x.replace(' p.a.', '') if isinstance(x, str) else x)
    df[col] = df[col].apply(lambda x: x.replace(' pa', '') if isinstance(x, str) else x)
    df[col] = df[col].apply(lambda x: x.replace('EUR ', '') if isinstance(x, str) else x)

non_percentage = ['tamaño_del_fondo', 'participaciones', 'dividendos_(últimos_12_meses)', 'historic_dividend_1 Año', 'historic_dividend_2023',	'historic_dividend_2022',	'historic_dividend_2021',	'historic_dividend_2020',	'historic_dividend_2019',	'historic_dividend_2015',	'historic_dividend_2014',	'historic_dividend_2013',	'historic_dividend_2016',	'historic_dividend_2018',	'historic_dividend_2017',	'historic_dividend_2012',	'historic_dividend_2011',]

percentages = [item for item in numeric if item not in non_percentage]

for col in percentages:
    if df[col].dtype != 'float':
        df[col] = df[col].str.rstrip('%').astype('float') / 100.0

df['top_10_holdings'] = df['top_10_holdings']/100

In [72]:
# Get datetimes

import locale
from datetime import datetime

def getDate(row):
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
    date_object = datetime.strptime(row, '%d de %B de %Y')
    return date_object

df['fecha_de_inicio/_de_cotización'] = df['fecha_de_inicio/_de_cotización'].apply(getDate)


In [73]:
# Explode category strings
def expand2(row):
    if row is None or isinstance(row, float):
        return {}
    else:
        return dict((f'{col}_{x}', True) for x in row)

string_col = ['foco_de_la_inversión', 'categoría']
categories = []

for col in string_col:
    df[col] = df[col].apply(lambda x: x.split(', ') if isinstance(x, str) else x)
    categories += df[col].to_list()
    df[col] = df[col].apply(expand2)
    temp_df = df[col].apply(pd.Series)
    df = df.drop([col], axis=1)

    df = pd.concat([df, temp_df], axis=1)

In [74]:
# Remove redundant columns and rename all columns
from unidecode import unidecode

cat2 = []
for cat in categories:
    if type(cat) != float:
        cat2 += cat
cat2 = set(cat2)

for cat in cat2:
    try:
        if df[df['categoría_'+cat] == True]['foco_de_la_inversión_'+cat].all():
            df = df.drop(['foco_de_la_inversión_'+cat], axis=1)
    except KeyError:
        pass

for col in df.columns:
    if 'foco_de_la_inversión_' in col or 'categoría' in col:
        cat = col.split('_')[-1]
        cat = ('_').join(cat.split(' ')).lower().strip()
        df = df.rename(columns={col: cat})
        col = cat
    df = df.rename(columns={col: unidecode(col).lower()})

In [75]:
# Rename country columns
for col in df.columns:
    if 'countries_' in col:
        part = col.split('_')
        part[-1] = ('_').join(part[-1].split(' '))
        part = ('_').join(part)
        df = df.rename(columns={col: part})

In [76]:
# just = df[['url',
#  'isin',
#  'indice',
#  'proveedor_de_fondo',
#  'fecha_de_inicio/_de_cotizacion',
#  'politica_de_distribucion',
#  'domicilio_del_fondo',
#  'ter',
#  'replicacion',
#  'riesgo_estrategico',
#  'sostenibilidad',
#  'divisa_del_fondo',
#  'tamano_del_fondo',
#  'participaciones',
#  'top_10_holdings',
#  'rentabilidad_actual_de_los_dividendos',
#  'frecuencia_de_distribucion',
#  'dividendos_(ultimos_12_meses)',
#  'volatilidad_1_ano_(in_eur)',
#  'sectors_otros',
#  'sectors_tecnologia',
#  'sectors_servicios financieros',
#  'sectors_salud',
#  'sectors_consumidor discrecional',
#  'sectors_materiales basicos',
#  'sectors_energia',
#  'sectors_industria',
#  'sectors_servicios publicos',
#  'sectors_bienes de consumo basicos',
#  'sectors_telecomunicaciones',
#  'sectors_inmobiliario',]]

just = df

---
#### ExtraETF
---

In [77]:
# Load extraETF json
with open('data/ibkr_extraetfs_scraped.json', 'r') as f:
    url_data = json.load(f)

    data_list = []
    for url in url_data:
        data = {}
        for prop in url:
            if type(prop) == str:
                data['url'] = prop
            elif type(prop) == list: ##### This will change if we assign stars
                value = prop[1:3]
                value[0] = value[0].split('-')[-1]
                if value[-1] == 'half':
                    value[-1] = '.5'
                else:
                    value[-1] = ''
                data['stars'] = float(('').join(value))
            else:
                data.update(prop)
        data_list.append(data)
    df = pd.DataFrame(data_list)

df['isin'] = df['url'].apply(lambda x: x.split('/')[-1])
print(df.shape)
df = df[['url', 'isin', 'stars', 'figura_del_índice', 'uso_de_los_ingresos', 'ter', 'tamaño_del_fondo', 'número_de_posiciones', 'td', 'porcentaje_del_top_10', 'moneda',  'estilo_de_ilustración', 'fecha_de_emisión_del_fondo', 'moneda_del_fondo', 'cobertura_de_divisas', 'domicilio_del_fondo', 'clasificación_sfdr', 'nombre_del_índice', 'valores_incluidos', 'posiciones_en_acciones', 'posiciones_en_bonos', 'posiciones_en_efectivo_y_otros', 'positions', 'top_10_positions','deuda_corporativa', 'efectivo', 'titulizado','tecnología', 'servicios_financieros', 'salud', 'industria', 'consumo_cíclico','deuda_pública',  'servicios_de_comunicación', 'defensivo', 'energía', 'materiales_básicos', 'servicios_públicos', 'bienes_inmobiliarios', '2024_dividend_value', '2024_dividend_percentage_of_distribution', '2023_dividend_value', '2023_dividend_percentage_of_distribution', '2022_dividend_value', '2022_dividend_percentage_of_distribution', '2021_dividend_value', '2021_dividend_percentage_of_distribution', '2020_dividend_value', '2020_dividend_percentage_of_distribution',   'estados_unidos', 'alemania', 'francia', 'china', 'japón',
       'reino_unido', 'países_bajos', 'italia', 'suiza', 'españa', 'taiwán',
       'australia', 'india', 'canadá', 'corea_del_sur', 'suecia', 'dinamarca',
       'bélgica', 'finlandia', 'brasil', 'hong_kong', 'singapur', 'sudáfrica',
       'austria', 'indonesia', 'arabia_saudita', 'turquía', 'méxico',
       'noruega', 'irlanda', 'polonia', 'portugal', 'xsn', 'tailandia',
       'emiratos_árabes_unidos', 'malasia', 'grecia', 'pública_municipal',
       'nueva_zelanda', 'catar', 'israel', 'hungría', 'omán', 'baréin',
       'luxemburgo', 'chequia', 'panamá', 'macao', 'chile', 'perú', 'rumanía',
       'colombia', 'egipto', 'eslovaquia', 'filipinas', 'kazajistán',
       'argentina', 'islas_caimanes', 'nigeria', 'puerto_rico', 'kuwait',
       'belice', 'marruecos', 'bermudas', 'jersey', 'islas_marshall',
       'república_de_chipre', 'guernsey', 'costa_rica',]]

(1949, 119)


In [78]:
# Clean up df
for col in df.columns:
    try:
        df[col] = df[col].replace('—', np.nan)
        df[col] = df[col].apply(lambda x: np.nan if len(x)==0 else x)
    except TypeError:
        continue

percentages = ['porcentaje_del_top_10', 'ter']

for col in percentages:
    df[col] = df[col]/100

/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_37400/3421880325.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace('—', np.nan)


In [79]:
# Translate date to datetime object
month_map = {
    'ene': 'enero',
    'feb': 'febrero',
    'mar': 'marzo',
    'abr': 'abril',
    'may': 'mayo',
    'jun': 'junio',
    'jul': 'julio',
    'ago': 'agosto',
    'sept': 'septiembre',
    'oct': 'octubre',
    'nov': 'noviembre',
    'dic': 'diciembre'
}


def getDate(row):
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
    day, month, year = row.split()
    month = month_map[month]
    row = f"{day} {month} {year}"
    date_object = datetime.strptime(row, '%d %B %Y')

    return date_object

df['fecha_de_emisión_del_fondo'] = df['fecha_de_emisión_del_fondo'].apply(getDate)


In [80]:
# Remove all column accents and prefix countries
from unidecode import unidecode

for col in df.columns:
    df = df.rename(columns={col: unidecode(col).lower()})

countries =  ['estados_unidos', 'alemania', 'francia', 'china', 'japon', 'reino_unido', 'paises_bajos', 'italia', 'suiza', 'espana', 'taiwan', 'australia', 'india', 'canada', 'corea_del_sur', 'suecia', 'dinamarca', 'belgica', 'finlandia', 'brasil', 'hong_kong', 'singapur', 'sudafrica', 'austria', 'indonesia', 'arabia_saudita', 'turquia', 'mexico', 'noruega', 'irlanda', 'polonia', 'portugal', 'xsn', 'tailandia', 'emiratos_arabes_unidos', 'malasia', 'grecia', 'publica_municipal', 'nueva_zelanda', 'catar', 'israel', 'hungria', 'oman', 'barein', 'luxemburgo', 'chequia', 'panama', 'macao', 'chile', 'peru', 'rumania', 'colombia', 'egipto', 'eslovaquia', 'filipinas', 'kazajistan', 'argentina', 'islas_caimanes', 'nigeria', 'puerto_rico', 'kuwait', 'belice', 'marruecos', 'bermudas', 'jersey', 'islas_marshall', 'republica_de_chipre', 'guernsey', 'costa_rica',]

for col in countries:
    df = df.rename(columns={col: 'countries_'+col})

---
#### Merge
---

In [81]:
# Rename columns to merge
df = df.rename(columns={'uso_de_los_ingresos'	: 'politica_de_distribucion',
'tamano_del_fondo' : 'tamano_del_fondo',
'número_de_posiciones' : 'participaciones',
'moneda' : 'divisa_del_fondo',
'fecha_de_emision_del_fondo' : 'fecha_de_inicio/_de_cotizacion',
'figura_del_indice' : 'replicacion',
'cobertura_de_divisas' : 'riesgo_de_divisa',
'domicilio_del_fondo' : 'domicilio_del_fondo',
'positions' : 'holdings',
'nombre_del_indice' : 'indice',
'porcentaje_del_top_10' : 'top_10_holdings',
'bienes_inmobiliarios' : 'sectors_inmobiliario',
'tecnologia' : 'sectors_tecnologia',
'servicios_financieros' : 'sectors_servicios financieros',
'salud' : 'sectors_salud',
'industria' : 'sectors_industria',
'servicios_de_comunicacion' : 'sectors_telecomunicaciones	',
'energia' : 'sectors_energia'	,
'materiales_basicos' : 'sectors_materiales basicos'	,
'servicios_públicos' : 'sectors_servicios publicos',
'2023_dividend_value' : 'historic_dividend_2023',
'2022_dividend_value' : 'historic_dividend_2022',
'2021_dividend_value' : 'historic_dividend_2021',
'2020_dividend_value' : 'historic_dividend_2020',
})

In [82]:
# Merge the dfs
df_reset = df.set_index('isin')
just_reset = just.set_index('isin')

df_aligned, just_aligned = df_reset.align(just_reset, join='outer', axis=1)
merge = just_aligned.combine_first(df_aligned)
merge = merge.drop(['exchanges'], axis=1)

# Reset the index if you want 'isin' to be a column again
merge = merge.reset_index()


In [83]:
# Drop duplicate rows and almost empty columns
merge = merge.drop_duplicates(subset=['isin'])

# nan_percentage = merge.isnull().mean().sort_values()
# columns_to_drop = nan_percentage[nan_percentage >= 0.18].index
# merge = merge.drop(columns_to_drop, axis=1)

# merge = merge.dropna(subset='ter')
# # [k for k,v in merge.isnull().mean().sort_values().items()]
# merge = merge[['isin',
#  'url',
#  'stars',
#  'ter',
#  'tamano_del_fondo',
#  'riesgo_estrategico',
#  'replicacion',
#  'estilo_de_ilustracion',
#  'politica_de_distribucion',
#  'indice',
#  'fecha_de_inicio/_de_cotizacion',
#  'proveedor_de_fondo',
#  'domicilio_del_fondo',
#  'sostenibilidad',
#  'valores_incluidos',
#  'posiciones_en_bonos',
#  'posiciones_en_acciones',
#  'posiciones_en_efectivo_y_otros',
#  'numero_de_posiciones',
#  'top_10_holdings',
#  'top_10_positions',
#  'holdings',
#  'riesgo_de_divisa',
#  ]]

In [85]:
# Match up values
merge['replicacion'] = merge['replicacion'].replace({'física': 'physical',
                                                     'Replicación física perfecta': 'physical', 
                                                     'sintética': 'synthetic', 
                                                     'Sintética': 'synthetic',
                                                     'Muestreo': 'sample',})
merge['politica_de_distribucion'] = merge['politica_de_distribucion'].replace({'Acumulación': 'accumulated',
                                                                                'acumulación' : 'accumulated',
                                                                                'dividendo' : 'distributed',
                                                                                'Distribución' : 'distributed',})
merge['riesgo_de_divisa'] = merge['riesgo_de_divisa'].replace({'no': False,
                                                                'Sin cobertura de divisas': False,
                                                               'sí(gbp)': True,
                                                               'sí(eur)': True,
                                                               'sí(usd)': True,
                                                               'sí(chf)': True,
                                                                'Cobertura de divisas': True,
                                                               })
merge['domicilio_del_fondo'] = merge['domicilio_del_fondo'].replace({'irland': 'Irlanda',
                                                                     'frankreich': 'Francia',
                                                                     'switzerland': 'Suiza',
                                                                     'luxemburg': 'Luxemburgo',
                                                                     'spain': 'España',
                                                                     'deutschland': 'Alemania',
                                                                     'jersey': 'Jersey',
                                                                     'liechtenstein': 'Liechtenstein',
                                                                     'niederlande': 'Paises Bajos',
                                                                     'schweden': 'Suecia',})

/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_37400/815737498.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merge['riesgo_de_divisa'] = merge['riesgo_de_divisa'].replace({'no': False,


In [86]:
# load ibkr data and merge it with scraped data
eur = pd.read_csv('data/eur.csv')

analisis = pd.merge(merge, eur, on='isin', how='left', suffixes=('', '_y'))
analisis.isna().mean()

analisis = analisis.drop_duplicates('ibkr-symbol')

analisis.to_csv('data/analisis.csv', index=False)

In [90]:
analisis.isna().mean().sort_values()

isin                           0.000000
exchange                       0.000000
region                         0.000000
ibkr-symbol                    0.000000
description                    0.000000
                                 ...   
sudafrica                      0.999512
countries_islas_marshall       0.999512
countries_belice               0.999512
silver_future_leverage_(3x)    0.999512
economia_circular              0.999512
Length: 322, dtype: float64

---
#### Historical
---

In [88]:
# Get historical data from ibkr
from ib_async import *
util.startLoop()

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=2)
# ib.reqMarketDataType(4)

remaining = [symb for symb in remaining if symb not in used_symbols]

for i, symbol in enumerate(remaining):
    try:
        contract = Stock(symbol, 'SMART', 'EUR')
        data = ib.reqHistoricalData(
            contract, 
            endDateTime='', 
            durationStr='3 Y',
            barSizeSetting='1 hour', 
            whatToShow= 'MIDPOINT', 
            useRTH=True)
        util.df(data).to_csv(f'data/1h_data/{symbol}.csv', index=False)
        used_symbols.append(symbol)
        print(f'{symbol} == {(i+1)} == {round(((i+1)/len(remaining))*100, 3)}%')
    except AttributeError as e:
        print(f'{symbol} == {e}')
        

ib.disconnect()


ModuleNotFoundError: No module named 'ib_async'

In [ ]:
# historical_df = pd.DataFrame(historical_data, columns=['ibkr-symbol', 'historical_data'])
# final = pd.merge(analisis, historical_df, on='ibkr-symbol')